# A model on frame data, with Candle

Polars data becomes a tensor, a small model scores every row, and the scores come back as a
column of the frame.

This notebook needs the **Candle kernel**: rnx with the Polars and Candle adapters assembled in
(see the README's "Candle notebooks" section). Nothing is downloaded: the data and the model are
bundled in `demos/data`. The model is tiny (3 inputs, 8 hidden units, 1 output) and its weights
are **fixed values chosen to show the plumbing, not trained on anything**, so the scores mean
nothing beyond the arithmetic.

Load the features: 64 rows of three numbers.

In [1]:
let df = polars::read_csv("../data/features.csv", [("id", "i64"), ("a", "f64"), ("b", "f64"), ("c", "f64")])?;
df

id,a,b,c
i64,f64,f64,f64
0,0.0,-3.0,0.0
1,0.25,-2.0,0.5
2,0.5,-1.0,1.0
3,0.75,0.0,0.125
4,1.0,1.0,0.625
…,…,…,…
59,14.75,0.0,0.625
60,15.0,1.0,1.125
61,15.25,2.0,0.25


The three feature columns as a 64 × 3 `f32` tensor. A tensor shows its dtype, its shape and
its first 8 × 8 corner.

In [2]:
let x = candle::Tensor::from_dense(df.to_dense(["a", "b", "c"], "f32")?)?;
x

,0,1,2
0,0.0,-3.0,0.0
1,0.25,-2.0,0.5
2,0.5,-1.0,1.0
3,0.75,0.0,0.125
4,1.0,1.0,0.625
5,1.25,2.0,1.125
6,1.5,3.0,0.25
7,1.75,-3.0,0.75
…,…,…,…


The model: two linear layers with a ReLU between them, loaded from a safetensors file.

In [3]:
let model = candle::Mlp::load("../data/mlp.safetensors")?;
model.dims()

(3, 8, 1)

Run it on the CPU: one score per row.

In [4]:
let y = model.forward(x)?;
y

,0
0,0.765625
1,0.640625
2,0.484375
3,0.51953125
4,0.36328125
5,0.40625
6,0.703125
7,0.9296875
…,…


The scores back in the frame, as a new column.

In [5]:
let scored = df.with_dense(y.to_dense(["score"])?)?;
scored

id,a,b,c,score
i64,f64,f64,f64,f32
0,0.0,-3.0,0.0,0.765625
1,0.25,-2.0,0.5,0.640625
2,0.5,-1.0,1.0,0.484375
3,0.75,0.0,0.125,0.51953125
4,1.0,1.0,0.625,0.36328125
…,…,…,…,…
59,14.75,0.0,0.625,1.4570313
60,15.0,1.0,1.125,1.3945313
61,15.25,2.0,0.25,1.2890625


The five highest-scoring rows.

In [6]:
let top = scored.sort(["score"], polars::SortMultipleOptions::new().with_order_descending(true))?;
top.head(Some(5))?

id,a,b,c,score
i64,f64,f64,f64,f32
63,15.75,-3.0,1.25,1.8203125
56,14.0,-3.0,0.5,1.6875
57,14.25,-2.0,1.0,1.625
49,12.25,-3.0,1.125,1.5976563
58,14.5,-1.0,0.125,1.5195313


In [7]:
let id = top.column("id")?.i64()?.get(0)?.unwrap();
let score = top.column("score")?.f32()?.get(0)?.unwrap();
println!("Highest score: row {} with {}, of {} rows scored", id, score, scored.height()?);

Highest score: row 63 with 1.8203125, of 64 rows scored
